# 第 7 周练习：游戏建议零售价（SRP）基准

## 练习目标（理念）

**发行商用例**：根据简短产品描述（版本、DLC、捆绑包、订阅等）预测游戏产品的**建议零售价 Suggested Retail Price (SRP)**。

本笔记本用 **零样本（zero-shot）** 对比两条推理路径，指标与课程微调评估同一套（MAE / RMSE / R²）：

1. **数据** — 小列表：产品描述 → 典型美元建议价（基础游戏、豪华版、DLC、季票、订阅、捆绑包等）
2. **OpenRouter** — 云端前沿小模型（零样本）
3. **Ollama** — 本地小模型（零样本）
4. **评估** — MAE、RMSE、R²

## 和本课第 7 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 价格预测基准 | 手工小测试集 + 指标函数 |
| 云端 Chat Completions | OpenRouter + `openai/gpt-4o-mini` |
| 本地 OpenAI 兼容接口 | Ollama `llama3.2:1b` |
| 从自由文本抽数字 | `extract_price` 正则 |

## 怎么跑

1. `.env` 准备 `OPENROUTER_API_KEY`
2. 本地：`ollama serve` 且已 `ollama pull llama3.2:1b`
3. 从上到下运行；最后看两条路径的 MAE/RMSE/R² 与样例预测


## 这对谁有用？

- **Publisher（发行商）** — 这是面向**发行方**的定价辅助工具：根据版本 / DLC / 捆绑包 / 订阅等描述，建议典型 SRP，方便企业**设定或核对价格**。
- 目标用户是定价决策者，而不是给玩家做「买不买」推荐。


In [ ]:
# ========== 导入 + OpenRouter / Ollama 两个 OpenAI 兼容客户端 ==========

# os：读环境变量（API Key）
import os
# re：从模型自由文本里用正则抽出价格数字
import re
# random：打乱测试集顺序（固定 seed 可复现）
import random
# load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# OpenAI：统一客户端；既可打 OpenRouter，也可打本地 Ollama
from openai import OpenAI
# numpy：向量化算 MAE / RMSE / R²
import numpy as np

# override=True：.env 覆盖已有环境变量（参数保持原样）
load_dotenv(override=True)

# ---------- OpenRouter（云端）----------
# OpenAI 兼容基址；URL 不可改译
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
# 从环境变量取 OpenRouter 密钥
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
# base_url 指向 OpenRouter，api_key 用上面读到的密钥
openrouter_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
# 云端模型 id（字符串必须保持原样）
FRONTIER_MODEL = "openai/gpt-4o-mini"

# ---------- Ollama（本地 OpenAI 兼容口）----------
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 本地模型名；需事先 ollama pull；注释里的 pull 命令保持英文可运行
OLLAMA_MODEL = "llama3.2:1b"  # Run: ollama pull llama3.2:1b
# 本地通常用占位 api_key="ollama"（Ollama 不校验，但客户端需要字段）
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
# 打印用标签：带上模型名，方便对照输出
OLLAMA_LABEL = f"Ollama ({OLLAMA_MODEL})"
print(f"Using {OLLAMA_LABEL}. Ensure Ollama is running (ollama serve) and model pulled (ollama pull {OLLAMA_MODEL}).")


## 游戏产品测试数据

每条：`summary`（英文描述，作为模型输入）→ `price`（美元典型建议价，作真值）。描述字符串是数据，不要翻译改写。


In [ ]:
# ========== 构造小测试集：描述 → 美元 SRP ==========

# 发给模型的问题模板（prompt 字符串不可改译，否则行为变）
QUESTION = "What is a typical suggested retail price in USD for this game product (to the nearest dollar)?"

# 游戏行业产品样例：版本 / DLC / 季票 / 订阅 / 捆绑包（典型建议零售价）
ITEMS = [
    {"summary": "AAA base game, standard edition, digital (PC/console).", "price": 59.99},
    {"summary": "Deluxe edition: base game + season pass + bonus cosmetics.", "price": 89.99},
    {"summary": "Story expansion DLC, ~10–15 hours, new region and quests.", "price": 24.99},
    {"summary": "Cosmetic DLC: character skin pack, 5 outfits.", "price": 4.99},
    {"summary": "Season pass only (all year-one DLC), sold separately.", "price": 29.99},
    {"summary": "Physical collector's edition: game, steelbook, artbook, statue.", "price": 129.0},
    {"summary": "Monthly subscription, full game library access, one platform.", "price": 14.99},
    {"summary": "Indie game, digital only, single-player, ~8 hours.", "price": 19.99},
    {"summary": "Premium in-game currency pack, mid-tier (e.g. 2000 coins).", "price": 9.99},
    {"summary": "Complete edition: base game + 2 story DLCs, digital bundle.", "price": 79.99},
]

# 固定随机种子，打乱顺序可复现
random.seed(42)
# 复制后再 shuffle，避免改到原始 ITEMS 列表对象本身以外的共享引用意外
test_data = ITEMS.copy()
random.shuffle(test_data)
print(f"Test set: {len(test_data)} game product items")


## 预测器与价格抽取

`extract_price`：从模型回复里抓第一个数字；两个 `predict_*`：分别走 OpenRouter 与 Ollama。


In [ ]:
# ========== 从自由文本抽价格 + 两条零样本预测函数 ==========

def extract_price(raw: str):
    """Parse first number (int or float) from model output; return None if not found."""
    # 空串或非 str：无法解析
    if not raw or not isinstance(raw, str):
        return None
    # 可选 $，再抓整数或小数；正则保持原样
    m = re.search(r"\$?\s*([0-9]+\.[0-9]*|[0-9]+)", raw.strip())
    if m:
        try:
            # 捕获组转 float
            return float(m.group(1))
        except ValueError:
            return None
    return None


def predict_openrouter(description: str) -> float:
    # 拼 user prompt：问题 + 产品描述 + 要求只回价格（英文指令不可改）
    prompt = f"{QUESTION}\n\nProduct: {description}\n\nReply with only the suggested price in dollars (e.g. 60 or 24.99)."
    # Chat Completions：model / messages / max_tokens 参数保持原样
    r = openrouter_client.chat.completions.create(
        model=FRONTIER_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=20,
    )
    # 取助手文本；None 当空串
    text = (r.choices[0].message.content or "").strip()
    out = extract_price(text)
    # 解析失败时返回 0.0，避免后面指标计算炸掉
    return out if out is not None else 0.0


def predict_ollama(description: str) -> float:
    # 防御：若客户端未建好则直接 0.0（本笔记本里通常已创建）
    if not ollama_client:
        return 0.0
    try:
        # 与 OpenRouter 同一套英文 prompt，保证对比公平
        prompt = f"{QUESTION}\n\nProduct: {description}\n\nReply with only the suggested price in dollars (e.g. 60 or 24.99)."
        r = ollama_client.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20,
        )
        text = (r.choices[0].message.content or "").strip()
        p = extract_price(text)
        return p if p is not None else 0.0
    except Exception as e:
        # 只打印一次错误，避免每条样本刷屏；错误文案保持英文（含可运行命令）
        if not getattr(predict_ollama, "_err_printed", False):
            predict_ollama._err_printed = True
            print(f"Ollama error: {type(e).__name__}: {e}. Is Ollama running? Try: ollama serve && ollama pull {OLLAMA_MODEL}")
        return 0.0

# 函数属性：标记「是否已打印过 Ollama 错误」
predict_ollama._err_printed = False


## 指标（MAE、RMSE、R²）与双路径对比

对同一 `test_data` 分别跑 OpenRouter 与 Ollama，打印三套指标。


In [ ]:
# ========== 评估函数 + 跑两条路径 ==========

def metrics(predictor, data):
    # 对每条样本的 summary 调预测器
    preds = [predictor(item["summary"]) for item in data]
    # 真值价格列表
    truths = [item["price"] for item in data]
    # 转 float 数组，便于向量运算
    preds = np.array(preds, dtype=float)
    truths = np.array(truths, dtype=float)
    # MAE：平均绝对误差
    mae = np.mean(np.abs(preds - truths))
    # RMSE：均方根误差
    rmse = np.sqrt(np.mean((preds - truths) ** 2))
    # R²：1 - SS_res/SS_tot；ss_tot=0 时退回 0.0
    ss_res = np.sum((truths - preds) ** 2)
    ss_tot = np.sum((truths - np.mean(truths)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    return {"MAE": mae, "RMSE": rmse, "R2": r2}


# OpenRouter 路径指标
m_or = metrics(predict_openrouter, test_data)
print(f"OpenRouter ({FRONTIER_MODEL}): MAE={m_or['MAE']:.1f}, RMSE={m_or['RMSE']:.1f}, R²={m_or['R2']:.3f}")

# Ollama 本地路径指标
m_ollama = metrics(predict_ollama, test_data)
print(f"{OLLAMA_LABEL}:  MAE={m_ollama['MAE']:.1f}, RMSE={m_ollama['RMSE']:.1f}, R²={m_ollama['R2']:.3f}")


In [ ]:
# ========== 抽前 4 条，并排看真值 vs 两模型预测 ==========

print("Sample predictions (first 4 game products):")
for item in test_data[:4]:
    # 同一描述分别问云端与本地
    p_or = predict_openrouter(item["summary"])
    p_ollama = predict_ollama(item["summary"])
    # 本地失败或 0.0 时用破折号占位，避免误读成「价格为零」
    ollama_str = "—" if (p_ollama is None or p_ollama == 0.0) else f"${p_ollama:.2f}"
    print(f"  True: ${item['price']:.2f} | OpenRouter: ${p_or:.2f} | {OLLAMA_LABEL}: {ollama_str}")
    # 描述截断，便于扫一眼
    print(f"    {item['summary'][:60]}...")
